In [1]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
PROJECT_PATH = Path.cwd().parent
print(PROJECT_PATH)
sys.path.append(str(PROJECT_PATH))


/Users/jenriquezafra/Proyectos/Dev/python/TFM/TFM


# Data generation


In [2]:
from src.solvers.heston_cos import COS_solver_scalar

This params are the same from the TFG provided from Kelly Muzaneza.

In [3]:
params_heston = [-0.5, 0.9, 0.1, 0.1, 0.36]
m = 1
K = 1
S0 = m*K
r = 0.1
tau = 1.0
COS_params = [64, 8]
iv_bounds = [1e-6, 5]
opt_type = "put"

V_test = COS_solver_scalar(params_Heston=params_heston, 
                           S0=S0, 
                           K=K, 
                           tau=tau,
                           r = r, 
                           COS_params=COS_params, 
                           opt_type=opt_type)
print(V_test)

0.15124638962628115


With $N=64$, the relative error is sufficientely small ($\sim 10^{-10}$)

#### Brent tests

In [4]:
from src.solvers.implied_vol import IV_Brent

### Sampling


In [6]:
from scipy.stats import qmc
N=10_000

sampler = qmc.LatinHypercube(d=5)
X = sampler.random(n=N)
print(X)

[[0.58023286 0.71992332 0.51083971 0.28879831 0.04941902]
 [0.89278119 0.64021834 0.55530254 0.30525218 0.07639577]
 [0.9562766  0.61213966 0.79101672 0.86738294 0.72326805]
 ...
 [0.5436917  0.74656988 0.19614213 0.11653394 0.05593878]
 [0.62581452 0.72386159 0.20395381 0.19929589 0.92439625]
 [0.4952099  0.39508736 0.44587845 0.84898528 0.15445593]]


### Synthetic data

In [8]:
data_path = PROJECT_PATH / "data" / "synth" / "1M_fixed" / "heston_synth_data.parquet"

synth_df = pd.read_parquet(data_path)
synth_df.head()

,rho,kappa,gamma,bar_v,v0,moneyness,tau,r,IV
0,-0.683106,0.9,0.161438,0.193119,0.36,1.131014,0.172470,0.003184,0.593600
1,-0.610020,0.9,0.652320,0.317308,0.36,1.331532,0.341642,0.003456,0.622732
2,-0.321588,0.9,0.270159,0.016827,0.36,0.703650,0.326931,0.045317,0.545998
3,-0.484129,0.9,0.264058,0.196835,0.36,0.929255,2.778686,0.026463,0.490582
4,-0.401914,0.9,0.388335,0.110392,0.36,1.275992,0.671800,0.020577,0.551245


#### LM root-finder


In [9]:
import os, sys
import numpy as np

root = os.path.abspath("..") if os.getcwd().endswith("notebooks") else os.getcwd()
if root not in sys.path:
    sys.path.append(root)

from src.solvers.implied_vol import IV_LM, IV_Brent
from src.solvers.heston_cos import COS_solver_scalar
from src.solvers.bs import BS_solver

# Params Heston
params_heston = np.array([-0.7, 1.5, 0.4, 0.04, 0.04])
S0, K, tau, r = 1.0, 1.0, 0.5, 0.01
COS_params = np.array([256, 10])  
opt_type = "put"

iv_lm = IV_LM(params_heston, S0, K, tau, r, COS_params, opt_type=opt_type, sigma0=0.2)
iv_brent = IV_Brent(params_heston, S0, K, tau, r, COS_params, opt_type=opt_type)

V_heston = COS_solver_scalar(params_heston, S0, K, tau, r, COS_params, opt_type=opt_type)
V_bs_lm = BS_solver(S0, K, tau, iv_lm, r, opt_type=opt_type)

print("iv_lm:", iv_lm)
print("iv_brent:", iv_brent)
print("abs(iv_lm - iv_brent):", abs(iv_lm - iv_brent))
print("price residual (BS_lm - Heston):", float(V_bs_lm - V_heston))


iv_lm: 0.18887835482173976
iv_brent: 0.18887835487548343
abs(iv_lm - iv_brent): 5.374367617605458e-11
price residual (BS_lm - Heston): 5.551115123125783e-17


---
# ANN
Made with pytorch

In [10]:
import torch
import torch.nn as nn

class toy_nn(nn.Module):
    """
    parecido al de Liu
    """

    def __init__(
            self,
            input_dim: int = 8,
            hidden_dims=(200, 200, 200, 200),
            output_dim: int = 1,
            dropout_rate: float = 0.0,
            initialization: str = "xavier_uniform",
    ):
        super().__init__()

        h1, h2, h3, h4 = hidden_dims
        self.fc1 = nn.Linear(input_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, h3)
        self.fc4 = nn.Linear(h3, h4)
        self.fc5 = nn.Linear(h4, output_dim)

        self.act = nn.ReLU()
        self.drop = nn.Dropout(p=dropout_rate)

        self._init_weights(initialization)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        x = self.drop(self.act(self.fc3(x)))
        x = self.drop(self.act(self.fc4(x)))
        x = self.fc5(x)         # linear output for regression
        return x
    
    def _init_weights(self, initialization: str) -> None:
        init = initialization.lower()

        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init == "xavier_uniform":
                    nn.init.xavier_uniform_(m.weight)
                elif init == "xavier_normal":
                    nn.init.xavier_normal_(m.weight)
                elif  init == "kaiming_uniform":
                    nn.init.kaiming_uniform_(m.weight)
                elif init == "kaiming_normal":
                    nn.init.kaiming_normal_(m.weight)
                else:
                    raise ValueError(f"Initialization:'{initialization}' not supported")
                
                nn.init.zeros_(m.bias)


In [11]:
# create the model
model = toy_nn(dropout_rate=0.0, initialization="xavier_normal")
print(model)

toy_nn(
  (fc1): Linear(in_features=8, out_features=200, bias=True)
  (fc2): Linear(in_features=200, out_features=200, bias=True)
  (fc3): Linear(in_features=200, out_features=200, bias=True)
  (fc4): Linear(in_features=200, out_features=200, bias=True)
  (fc5): Linear(in_features=200, out_features=1, bias=True)
  (act): ReLU()
  (drop): Dropout(p=0.0, inplace=False)
)


### NN training

In [14]:
from torch.utils.data import DataLoader, TensorDataset, random_split

# Take the synth data
df_X = synth_df.iloc[:, :-1]
df_y = synth_df.iloc[:, -1]

X = torch.from_numpy(df_X.values).float()
y = torch.from_numpy(df_y.values).float().view(-1, 1)

device = "mps" if torch.backends.mps.is_available() else "cpu"

# --- split train/val ---
dataset = TensorDataset(X, y)
N = len(dataset)
n_train = int(0.8 * N)
n_val = N - n_train # TODO: realmente debería ser solo el 10%
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

batch_size = 1024
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

# --- model ---
model = toy_nn(
    input_dim=8,
    hidden_dims=(200, 200, 200, 200),
    output_dim=1, 
    dropout_rate=0.0,
    initialization="xavier_uniform",
).to(device)

# --- loss + optimizer ---
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) 

# --- training with validation ---
epochs = 100

for epoch in range(1, epochs+1):
    # train
    model.train()
    train_sum = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

        train_sum += loss.item() * xb.size(0)
    
    train_mse = train_sum / n_train

    # validation
    model.eval()
    val_sum = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_sum += loss_fn(pred, yb).item() * xb.size(0)

    val_mse = val_sum / n_val

    if epoch == 1 or epoch%5 == 0:
        print(f"Epoch {epoch:3d}/{epochs} | train MSE: {train_mse:.6f} | val MSE: {val_mse:.6f}")


Epoch   1/100 | train MSE: 0.001204 | val MSE: 0.000007


KeyboardInterrupt: 

## Metrics from the training

In [13]:
metrics_dir = PROJECT_PATH / "outputs" / "runs" / "2026-01-05_16:07:55" / "metrics" / "metrics.parquet"
metric_df = pd.read_parquet(metrics_dir, engine="pyarrow")

plt.figure(figsize=(12,8))
plt.semilogy(metric_df["epoch"], metric_df["train_loss"], label="Train")
plt.semilogy(metric_df["epoch"], metric_df["val_loss"], label="Validation")
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.legend()
plt.grid(True, which="major")
plt.grid(True, which="minor", alpha=0.3)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jenriquezafra/Proyectos/Dev/python/TFM/TFM/outputs/runs/2026-01-05_16:07:55/metrics/metrics.parquet'

## Loading the trained model from a checkpoint

primero reconstruimos el modelo (la arquiitectura solo)

In [16]:
from src.models.ANN_pricer import ANN
device = "cpu"

model = ANN(
    input_dim=8,
    hidden_dims=(200, 200, 200, 200),
    output_dim=1,
    activation="relu",
    dropout_rate=0.0,
    initialization="xavier_uniform"
).to(device)


luego cargamos los pesos guardados tras entrenar

In [ ]:
# path to the checkpoint
ckpt_path = PROJECT_PATH / "outputs" / "runs" / "2026-01-06_20:22:01" / "checkpoints" / "model_best.pt"

# load the checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])

# eval mode
model.eval()

ANN(
  (fc1): Linear(in_features=8, out_features=200, bias=True)
  (fc2): Linear(in_features=200, out_features=200, bias=True)
  (fc3): Linear(in_features=200, out_features=200, bias=True)
  (fc4): Linear(in_features=200, out_features=200, bias=True)
  (fc5): Linear(in_features=200, out_features=1, bias=True)
  (act): ReLU()
  (drop): Dropout(p=0.0, inplace=False)
)

In [19]:
ckpt.keys()

dict_keys(['epoch', 'train_loss', 'val_loss', 'loss_name', 'model_state', 'optimizer_state'])